# Person-level model tuning

Tune **Isolation Forest**, **One-Class SVM**, and **LOF** on client-level features (`config.FEATURES`), same grain as `models/models.py` / `compare_models`.

**Objective:** maximize **precision@100** — among all persons, take the top 100 by anomaly score (higher = more anomalous); **precision@100** is (frauds in that top set) / 100, same definition as in `models/fit_and_evaluate.py`. **recall@100** is still logged for comparison.

In [1]:
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

ROOT = Path.cwd().parent if Path.cwd().name == "A_tuning" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import load_data, FEATURES

from models.scaling import scale_features

# Person-level defaults aligned with `models/models.py` `compare_models`.
ACTIVITY_STATE = 1
DAYS_VISITS = 2
PERSON_FEATURES = list(FEATURES)

EXCLUDE_FRAUD_FROM_TRAINING = True
MAX_OCSVM_TRAIN = 4000
RANDOM_STATE = 42

_, client_data, _, _ = load_data(
    activity_state=ACTIVITY_STATE,
    days_visits=DAYS_VISITS,
)
y_fraud = client_data["is_fraud"].astype(int).values
n_fraud = int(y_fraud.sum())
n_total = len(client_data)
print(f"Samples: {n_total} | Known fraud persons: {n_fraud}")
print(f"Features ({len(PERSON_FEATURES)}): {PERSON_FEATURES}")

if EXCLUDE_FRAUD_FROM_TRAINING:
    train_mask = y_fraud == 0
    fit_subset = client_data.loc[train_mask]
    X_fit_df, X_eval_df = scale_features(
        data=client_data,
        features=PERSON_FEATURES,
        scaler_type="standard",
        fit_data=fit_subset,
    )
    X_fit = X_fit_df.to_numpy(dtype=np.float64, copy=False)
    X_eval = X_eval_df.to_numpy(dtype=np.float64, copy=False)
else:
    X_full = scale_features(
        data=client_data,
        features=PERSON_FEATURES,
        scaler_type="standard",
    )
    X_fit = X_eval = X_full.to_numpy(dtype=np.float64, copy=False)

n_fit = len(X_fit)

K_OPT = 100  # optimize recall @ this rank cutoff


def recall_at_k(anomaly_scores: np.ndarray, fraud_binary: np.ndarray, k: int = K_OPT) -> float:
    """Higher anomaly_scores = more anomalous (same convention as fit_and_evaluate)."""
    fraud_binary = np.asarray(fraud_binary).astype(int)
    n_fraud = int(fraud_binary.sum())
    if n_fraud == 0:
        return float("nan")
    order = np.argsort(-anomaly_scores)
    fraud_mask = fraud_binary.astype(bool)
    top_k = order[: min(k, len(order))]
    return float(fraud_mask[top_k].sum() / n_fraud)


def precision_at_k(anomaly_scores: np.ndarray, fraud_binary: np.ndarray, k: int = K_OPT) -> float:
    """Higher anomaly_scores = more anomalous; precision@k = (frauds in top k) / k (fit_and_evaluate convention)."""
    fraud_binary = np.asarray(fraud_binary).astype(int)
    order = np.argsort(-anomaly_scores)
    fraud_mask = fraud_binary.astype(bool)
    top_k = order[: min(k, len(order))]
    return float(fraud_mask[top_k].sum() / k)


def lof_n_neighbors(requested: int, n_fit_samples: int) -> int:
    return int(min(max(5, requested), max(1, n_fit_samples - 1)))

Samples: 197763 | Known fraud persons: 43
Features (18): ['bonus_trn_count', 'share_top_waiter', 'share_bonuses_used_top_waiter', 'share_top_places', 'num_of_trn_prcnt', 'days_visits_prcnt', 'gross_amount_mean_prcnt', 'gross_amount_sum_prcnt', 'bonuses_accum_sum_prcnt', 'bonuses_used_sum_prcnt', 'num_of_waiters_prcnt', 'gross_amount_max_prcnt', 'first_last_trn_diff_prcnt', 'first_second_trn_diff_prcnt', 'first_third_trn_diff_prcnt', 'time_between_trn_median_prcnt', 'trn_per_day_prcnt', 'num_of_places_prcnt']


### Isolation Forest

Grid over `n_estimators`, `contamination`, `max_samples`. Initializes `rows`. Metrics: **precision@100** (primary), **recall@100** from anomaly scores (`-score_samples`).

In [ ]:
rows = []

for n_est, cont, ms in product(
    [100, 200, 500],
    [0.0005],
    [1.0, 0.8],
):
    try:
        iso = IsolationForest(
            n_estimators=n_est,
            contamination=cont,
            max_samples=ms,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        iso.fit(X_fit)
        scores = -iso.score_samples(X_eval)
        rows.append(
            {
                "model": "IsolationForest",
                "precision@100": precision_at_k(scores, y_fraud),
                "recall@100": recall_at_k(scores, y_fraud),
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "IsolationForest",
                "precision@100": np.nan,
                "recall@100": np.nan,
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
                "error": str(exc),
            }
        )

print(f"Isolation Forest: {sum(1 for r in rows if r['model'] == 'IsolationForest')} fits, rows total = {len(rows)}")

Isolation Forest: 1 fits, rows total = 1


### One-Class SVM

Subsample training to `MAX_OCSVM_TRAIN` when needed (same as `fit_and_evaluate`). Grid includes **`kernel`** (`rbf`, `linear`, `poly`, `sigmoid`). Metrics: **precision@100** (primary), **recall@100** from `-decision_function`.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
if n_fit > MAX_OCSVM_TRAIN:
    idx_sub = rng.choice(n_fit, size=MAX_OCSVM_TRAIN, replace=False)
    X_ocsvm_fit = X_fit[idx_sub]
else:
    X_ocsvm_fit = X_fit

for nu, gamma, kernel in product(
    [0.0002, 0.0005, 0.001, 0.005, 0.01],
    ["scale", 0.001, 0.01, 0.1],
    ["rbf", "linear"],
):
    try:
        ocsvm = OneClassSVM(kernel=kernel, nu=nu, gamma=gamma)
        ocsvm.fit(X_ocsvm_fit)
        scores = -ocsvm.decision_function(X_eval)
        rows.append(
            {
                "model": "OneClassSVM",
                "precision@100": precision_at_k(scores, y_fraud),
                "recall@100": recall_at_k(scores, y_fraud),
                "nu": nu,
                "gamma": gamma,
                "kernel": kernel,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "OneClassSVM",
                "precision@100": np.nan,
                "recall@100": np.nan,
                "nu": nu,
                "gamma": gamma,
                "kernel": kernel,
                "error": str(exc),
            }
        )

print(f"One-Class SVM: {sum(1 for r in rows if r['model'] == 'OneClassSVM')} fits, rows total = {len(rows)}")

One-Class SVM: 184 fits, rows total = 190


### LOF (Local Outlier Factor)

`novelty=True`: fit on `X_fit`, score on `X_eval`. Metrics: **precision@100** (primary), **recall@100** from `-score_samples`. Sweeps **`n_neighbors`** (each clamped to `[5, n_fit - 1]`).

In [ ]:
# # n_neighbors is clamped in lof_n_neighbors (min 5, max n_fit - 1)
# for nn_req, cont in product(
#     # [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 2000, 5000, 10000],
#     [100, 150, 200, 300, 500],
#     [0.01],
# ):
#     nn = lof_n_neighbors(nn_req, n_fit)
#     try:
#         lof = LocalOutlierFactor(
#             n_neighbors=nn,
#             contamination=cont,
#             metric="minkowski",
#             p=2,
#             novelty=True,
#         )
#         lof.fit(X_fit)
#         scores = -lof.score_samples(X_eval)
#         rows.append(
#             {
#                 "model": "LOF",
#                 "recall@100": recall_at_k(scores, y_fraud),
#                 "n_neighbors_requested": nn_req,
#                 "n_neighbors": nn,
#                 "contamination": cont,
#             }
#         )
#         print('results for LOF with n_neighbors =', nn_req, 'and contamination =', cont, 'are:', rows[-1])
#     except Exception as exc:
#         rows.append(
#             {
#                 "model": "LOF",
#                 "recall@100": np.nan,
#                 "n_neighbors_requested": nn_req,
#                 "n_neighbors": nn,
#                 "contamination": cont,
#                 "error": str(exc),
#             }
#         )

# print(f"LOF: {sum(1 for r in rows if r['model'] == 'LOF')} fits, rows total = {len(rows)}")

results for LOF with n_neighbors = 100 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 100, 'n_neighbors': 100, 'contamination': 0.01}
results for LOF with n_neighbors = 150 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 150, 'n_neighbors': 150, 'contamination': 0.01}
results for LOF with n_neighbors = 200 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 200, 'n_neighbors': 200, 'contamination': 0.01}
results for LOF with n_neighbors = 300 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 300, 'n_neighbors': 300, 'contamination': 0.01}
results for LOF with n_neighbors = 500 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 500, 'n_neighbors': 500, 'contamination': 0.01}
LOF: 5 fits, rows total = 5


results for LOF with n_neighbors = 100 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 100, 'n_neighbors': 100, 'contamination': 0.01}  
results for LOF with n_neighbors = 150 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 150, 'n_neighbors': 150, 'contamination': 0.01}  
results for LOF with n_neighbors = 200 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.09302325581395349, 'n_neighbors_requested': 200, 'n_neighbors': 200, 'contamination': 0.01}  
results for LOF with n_neighbors = 300 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 300, 'n_neighbors': 300, 'contamination': 0.01}  
results for LOF with n_neighbors = 500 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 500, 'n_neighbors': 500, 'contamination': 0.01}  
LOF: 5 fits, rows total = 5

In [ ]:
# # n_neighbors is clamped in lof_n_neighbors (min 5, max n_fit - 1)
# for nn_req, cont in product(
#     # [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 2000, 5000, 10000],
#     [750, 1000, 2000, 5000, 10000],
#     [0.01],
# ):
#     nn = lof_n_neighbors(nn_req, n_fit)
#     try:
#         lof = LocalOutlierFactor(
#             n_neighbors=nn,
#             contamination=cont,
#             metric="minkowski",
#             p=2,
#             novelty=True,
#         )
#         lof.fit(X_fit)
#         scores = -lof.score_samples(X_eval)
#         rows.append(
#             {
#                 "model": "LOF",
#                 "precision@100": precision_at_k(scores, y_fraud),
#                 "recall@100": recall_at_k(scores, y_fraud),
#                 "n_neighbors_requested": nn_req,
#                 "n_neighbors": nn,
#                 "contamination": cont,
#             }
#         )
#         print('results for LOF with n_neighbors =', nn_req, 'and contamination =', cont, 'are:', rows[-1])
#     except Exception as exc:
#         rows.append(
#             {
#                 "model": "LOF",
#                 "precision@100": np.nan,
#                 "recall@100": np.nan,
#                 "n_neighbors_requested": nn_req,
#                 "n_neighbors": nn,
#                 "contamination": cont,
#                 "error": str(exc),
#             }
#         )

# print(f"LOF: {sum(1 for r in rows if r['model'] == 'LOF')} fits, rows total = {len(rows)}")

results for LOF with n_neighbors = 750 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 750, 'n_neighbors': 750, 'contamination': 0.01}
results for LOF with n_neighbors = 1000 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.11627906976744186, 'n_neighbors_requested': 1000, 'n_neighbors': 1000, 'contamination': 0.01}
results for LOF with n_neighbors = 2000 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.20930232558139536, 'n_neighbors_requested': 2000, 'n_neighbors': 2000, 'contamination': 0.01}
results for LOF with n_neighbors = 5000 and contamination = 0.01 are: {'model': 'LOF', 'recall@100': 0.3953488372093023, 'n_neighbors_requested': 5000, 'n_neighbors': 5000, 'contamination': 0.01}


: 

### Summary

Build `tuning_results` from all `rows` and rank by **`precision@100`** (tie-break with **`recall@100`**). Run the three model code cells in order (Isolation Forest first — it initializes `rows`).

In [3]:
tuning_results = pd.DataFrame(rows)
metric_primary = "precision@100"
metric_secondary = "recall@100"
print(
    f"Finished {len(tuning_results)} fits (primary: {metric_primary}, secondary: {metric_secondary}).\n"
)

for model_name in ["IsolationForest", "OneClassSVM", "LOF"]:
    sub = tuning_results[tuning_results["model"] == model_name].sort_values(
        [metric_primary, metric_secondary],
        ascending=[False, False],
        na_position="last",
    )
    print(f"=== Top settings: {model_name} (by {metric_primary}, then {metric_secondary}) ===")
    print(sub.head(8).to_string(index=False))
    print()

best = tuning_results.sort_values(
    [metric_primary, metric_secondary],
    ascending=[False, False],
    na_position="last",
).iloc[0]
print(f"Best overall by {metric_primary} (tie-break {metric_secondary}):")
print(best.to_string())

Finished 1 fits (primary: precision@100, secondary: recall@100).

=== Top settings: IsolationForest (by precision@100, then recall@100) ===
          model  precision@100  recall@100  n_estimators  contamination  max_samples
IsolationForest           0.27    0.627907           200         0.0005          1.0

=== Top settings: OneClassSVM (by precision@100, then recall@100) ===
Empty DataFrame
Columns: [model, precision@100, recall@100, n_estimators, contamination, max_samples]
Index: []

=== Top settings: LOF (by precision@100, then recall@100) ===
Empty DataFrame
Columns: [model, precision@100, recall@100, n_estimators, contamination, max_samples]
Index: []

Best overall by precision@100 (tie-break recall@100):
model            IsolationForest
precision@100               0.27
recall@100              0.627907
n_estimators                 200
contamination             0.0005
max_samples                  1.0
